# SVD y PCA desde cero con NumPy

En el notebook anterior aprendimos álgebra lineal con NumPy: transpuestas, productos, eigenvalores, y terminamos con SVD. Ahora vamos a llevar esas herramientas a dos aplicaciones reales que aparecen constantemente en ciencia de datos y machine learning:

1. **Compresión de imágenes con SVD** — representar una imagen con mucho menos datos sin perder (casi) nada de calidad
2. **PCA desde cero** — reducir la dimensionalidad de un dataset construyendo el algoritmo con puras operaciones de NumPy, sin librerías de ML

La idea central de ambas es la misma: **no toda la información en tus datos es igualmente importante**. SVD y PCA encuentran las direcciones donde hay más "acción" y descartan el resto.

Los temas que cubriremos:

1. Repaso rápido: qué nos dice SVD
2. SVD para comprimir una imagen en escala de grises
3. Compresión de imagen a color (RGB)
4. Relación calidad vs. compresión
5. De SVD a PCA — la conexión
6. PCA desde cero, paso a paso
7. Varianza explicada y el scree plot
8. Visualización 2D del espacio latente
9. Reconstrucción de dígitos desde componentes principales
10. Comparación con scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from skimage import data as skdata
from sklearn.datasets import load_digits

# Para que las imágenes se vean en el notebook
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['image.cmap'] = 'gray'

np.set_printoptions(precision=4, suppress=True)
print(f"NumPy version: {np.__version__}")

---
# 1. Repaso rápido: qué nos dice SVD

Recuerda: cualquier matriz $A$ de shape $(m \times n)$ se puede descomponer exactamente como:

$$A = U \Sigma V^T$$

- $U$ → columnas son las **direcciones** en el espacio de filas (eigenvectores de $AA^T$)
- $\Sigma$ → **valores singulares** $\sigma_1 \geq \sigma_2 \geq \ldots \geq 0$, ordenados de mayor a menor
- $V^T$ → filas son las **direcciones** en el espacio de columnas

La clave es que podemos reconstruir $A$ de forma aproximada usando solo los primeros $k$ valores singulares:

$$A_k = \sum_{i=1}^{k} \sigma_i \, \mathbf{u}_i \mathbf{v}_i^T$$

Cada término $\sigma_i \mathbf{u}_i \mathbf{v}_i^T$ es una matriz de rango 1. Los primeros términos capturan la mayor parte de la "energía" de la matriz. Los últimos, ruido o detalles finos.

In [ ]:
# Visualicemos los valores singulares de una imagen real
# para entender intuitivamente cuánta información captura cada uno

img_gray = skdata.camera().astype(float)   # 512x512, escala de grises
print("Imagen shape:", img_gray.shape)
print("Dtype:", img_gray.dtype)

U, S, Vt = np.linalg.svd(img_gray, full_matrices=False)
print(f"\nU  shape: {U.shape}")
print(f"S  shape: {S.shape}   ← {len(S)} valores singulares")
print(f"Vt shape: {Vt.shape}")

In [ ]:
# Los valores singulares caen rápido — eso es lo que hace posible la compresión
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(S, color='steelblue')
axes[0].set_title('Valores singulares (escala lineal)')
axes[0].set_xlabel('Índice')
axes[0].set_ylabel('σ')
axes[0].axvline(x=50, color='red', linestyle='--', alpha=0.6, label='k=50')
axes[0].legend()

axes[1].semilogy(S, color='steelblue')
axes[1].set_title('Valores singulares (escala logarítmica)')
axes[1].set_xlabel('Índice')
axes[1].set_ylabel('σ (log)')
axes[1].axvline(x=50, color='red', linestyle='--', alpha=0.6, label='k=50')
axes[1].legend()

plt.tight_layout()
plt.show()

# Energía capturada por los primeros k valores singulares
energia_total = np.sum(S**2)
for k in [5, 10, 20, 50, 100]:
    energia_k = np.sum(S[:k]**2) / energia_total * 100
    print(f"  k={k:>3}  →  {energia_k:.1f}% de la energía total")

Con solo 50 de los 512 valores singulares ya capturamos más del 90% de la energía de la imagen. Eso es la intuición detrás de la compresión.

---
# 2. SVD para Comprimir una Imagen en Escala de Grises

Una imagen en escala de grises es simplemente una matriz 2D donde cada elemento es la intensidad de un pixel (0 = negro, 255 = blanco). SVD nos permite representar esa matriz con muchos menos números.

In [ ]:
def comprimir_svd(imagen, k):
    """
    Recibe una imagen 2D (escala de grises) y un rango k.
    Regresa la imagen reconstruida usando solo los primeros k
    valores singulares.
    """
    U, S, Vt = np.linalg.svd(imagen, full_matrices=False)

    # Reconstrucción: U[:, :k] @ diag(S[:k]) @ Vt[:k, :]
    # Que es equivalente a: (U[:, :k] * S[:k]) @ Vt[:k, :]
    # (Broadcasting — multiplicar cada columna de U por el escalar correspondiente de S)
    reconstruida = (U[:, :k] * S[:k]) @ Vt[:k, :]

    # Clipping para que los valores queden en rango válido
    return np.clip(reconstruida, 0, 255)


def ratio_compresion(imagen, k):
    """
    Calcula el ratio de compresión.
    Original: m*n valores.
    Comprimido: k*(m + n + 1) valores (U, S, Vt truncados).
    """
    m, n = imagen.shape
    valores_originales    = m * n
    valores_comprimidos   = k * (m + n + 1)
    return valores_originales / valores_comprimidos


print("Funciones definidas ✓")

In [ ]:
# Visualizamos la reconstrucción con distintos valores de k
ks = [1, 5, 15, 50, 100, 512]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.ravel()

for i, k in enumerate(ks):
    if k == 512:
        img_rec = img_gray   # original completo
        titulo  = f"Original (k=512)"
    else:
        img_rec = comprimir_svd(img_gray, k)
        ratio   = ratio_compresion(img_gray, k)
        titulo  = f"k={k}  |  ratio {ratio:.1f}x"

    axes[i].imshow(img_rec, vmin=0, vmax=255)
    axes[i].set_title(titulo, fontsize=11)
    axes[i].axis('off')

plt.suptitle('Compresión SVD — imagen en escala de grises', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Tabla de compresión con métricas numéricas
# MSE y PSNR son las métricas estándar de calidad de imagen

def mse(original, reconstruida):
    return np.mean((original - reconstruida) ** 2)

def psnr(original, reconstruida):
    """Peak Signal-to-Noise Ratio — más alto es mejor. >30 dB se considera buena calidad."""
    error = mse(original, reconstruida)
    if error == 0:
        return float('inf')
    return 10 * np.log10(255**2 / error)

print(f"{'k':>6}  {'ratio':>7}  {'% energía':>10}  {'MSE':>8}  {'PSNR (dB)':>10}")
print("-" * 50)

energia_total = np.sum(np.linalg.svd(img_gray, compute_uv=False)**2)
_, S_full, _ = np.linalg.svd(img_gray, full_matrices=False)

for k in [1, 5, 10, 20, 50, 100, 200, 512]:
    if k == 512:
        rec     = img_gray
        ratio   = 1.0
        energia = 100.0
    else:
        rec     = comprimir_svd(img_gray, k)
        ratio   = ratio_compresion(img_gray, k)
        energia = np.sum(S_full[:k]**2) / energia_total * 100

    error = mse(img_gray, rec)
    calidad = psnr(img_gray, rec)
    print(f"{k:>6}  {ratio:>7.1f}x  {energia:>9.1f}%  {error:>8.2f}  {calidad:>10.2f}")

> Con k=50 la imagen ocupa ~5x menos espacio que el original y tiene un PSNR de más de 30 dB — umbral donde el ojo humano generalmente no percibe degradación.

---
# 3. Compresión de Imagen a Color (RGB)

Una imagen a color tiene **3 canales** (R, G, B), cada uno es una matriz 2D. La estrategia es simple: aplicar SVD de forma independiente a cada canal y reconstruir.

In [ ]:
img_color = skdata.astronaut().astype(float)   # 512x512x3
print("Imagen color shape:", img_color.shape)
print("Canales: R, G, B")

plt.figure(figsize=(4, 4))
plt.imshow(img_color.astype(np.uint8))
plt.title('Imagen original (astronauta)')
plt.axis('off')
plt.show()

In [ ]:
def comprimir_rgb(imagen, k):
    """
    Aplica compresión SVD canal por canal a una imagen RGB.
    """
    reconstruida = np.zeros_like(imagen)
    for canal in range(3):                              # 0=R, 1=G, 2=B
        reconstruida[:, :, canal] = comprimir_svd(imagen[:, :, canal], k)
    return np.clip(reconstruida, 0, 255).astype(np.uint8)


# Visualizamos los canales individualmente
nombres_canales = ['Canal R (rojo)', 'Canal G (verde)', 'Canal B (azul)']
colormaps = ['Reds_r', 'Greens_r', 'Blues_r']

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for i, (nombre, cmap) in enumerate(zip(nombres_canales, colormaps)):
    axes[i].imshow(img_color[:, :, i], cmap=cmap)
    axes[i].set_title(nombre)
    axes[i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Compresión a diferentes k — imagen a color
ks_color = [2, 10, 30, 80]

fig, axes = plt.subplots(1, 5, figsize=(15, 4))

# Original
axes[0].imshow(img_color.astype(np.uint8))
axes[0].set_title('Original', fontsize=10)
axes[0].axis('off')

for i, k in enumerate(ks_color):
    rec   = comprimir_rgb(img_color, k)
    ratio = ratio_compresion(img_color[:, :, 0], k)
    axes[i+1].imshow(rec)
    axes[i+1].set_title(f'k={k}\n{ratio:.1f}x más pequeño', fontsize=10)
    axes[i+1].axis('off')

plt.suptitle('Compresión SVD — imagen RGB (astronauta)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
# 4. Relación Calidad vs. Compresión

¿Cómo elegir k? Depende del trade-off que quieras. Vamos a graficar la curva PSNR vs. ratio de compresión para tener una vista completa.

In [ ]:
ks_rango = list(range(1, 30)) + list(range(30, 100, 5)) + list(range(100, 513, 20))

ratios  = []
psnrs   = []
energias = []

for k in ks_rango:
    rec = comprimir_svd(img_gray, min(k, 512))
    ratios.append(ratio_compresion(img_gray, k))
    psnrs.append(psnr(img_gray, rec))
    energias.append(np.sum(S_full[:k]**2) / energia_total * 100)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# PSNR vs k
axes[0].plot(ks_rango, psnrs, color='steelblue')
axes[0].axhline(y=30, color='red', linestyle='--', alpha=0.7, label='30 dB (buena calidad)')
axes[0].axhline(y=40, color='green', linestyle='--', alpha=0.7, label='40 dB (excelente)')
axes[0].set_xlabel('k (componentes SVD)')
axes[0].set_ylabel('PSNR (dB)')
axes[0].set_title('Calidad de reconstrucción vs. k')
axes[0].legend()
axes[0].grid(alpha=0.3)

# PSNR vs ratio de compresión
axes[1].plot(ratios, psnrs, color='darkorange')
axes[1].axhline(y=30, color='red', linestyle='--', alpha=0.7, label='30 dB')
axes[1].set_xlabel('Ratio de compresión (veces más pequeño)')
axes[1].set_ylabel('PSNR (dB)')
axes[1].set_title('Trade-off calidad vs. compresión')
axes[1].invert_xaxis()   # más compresión a la izquierda
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Encontrar el k mínimo para alcanzar 30 dB
k_30db = ks_rango[next(i for i, p in enumerate(psnrs) if p >= 30)]
print(f"k mínimo para PSNR ≥ 30 dB: k={k_30db}  (ratio {ratio_compresion(img_gray, k_30db):.1f}x)")

---
# 5. De SVD a PCA — La Conexión

PCA y SVD no son cosas distintas. PCA **es** SVD aplicado a la matriz de datos centrada.

Dado un dataset $X$ de shape $(n\_muestras \times n\_features)$:

1. Centramos: $\tilde{X} = X - \bar{X}$ (restar la media de cada feature)
2. Calculamos la **matriz de covarianza**: $C = \frac{1}{n-1} \tilde{X}^T \tilde{X}$
3. Los eigenvectores de $C$ son los **componentes principales**
4. Los eigenvalores de $C$ dicen cuánta varianza explica cada componente

El truco es que no necesitamos construir $C$ explícitamente. Si hacemos SVD de $\tilde{X}$:

$$\tilde{X} = U \Sigma V^T$$

entonces $V$ (las filas de $V^T$) **son exactamente los componentes principales** y los eigenvalores de $C$ son $\frac{\sigma_i^2}{n-1}$.

Es más estable numéricamente hacer SVD de $\tilde{X}$ que eigendescomposición de $C$, por eso así lo implementa scikit-learn.

In [ ]:
# Verifiquemos la conexión con un ejemplo pequeño
np.random.seed(42)
X_small = np.random.randn(100, 5)   # 100 muestras, 5 features

# Método 1: eigendescomposición de la matriz de covarianza
X_c = X_small - X_small.mean(axis=0)
C   = (X_c.T @ X_c) / (len(X_c) - 1)   # matriz de covarianza
eigenvalores, eigenvectores = np.linalg.eigh(C)  # eigh para matrices simétricas
# eigh ordena de menor a mayor — invertimos
eigenvectores = eigenvectores[:, ::-1]

# Método 2: SVD de X centrada
U, S, Vt = np.linalg.svd(X_c, full_matrices=False)
componentes_svd = Vt   # filas de Vt son los componentes principales

# Los componentes deben ser iguales (salvo signo — los eigenvectores son únicos salvo flip)
print("Primer componente (cov):  ", np.round(eigenvectores[:, 0], 4))
print("Primer componente (SVD):  ", np.round(Vt[0], 4))
print()
print("¿Son iguales (salvo signo)?",
      np.allclose(np.abs(eigenvectores[:, 0]), np.abs(Vt[0])))

---
# 6. PCA desde Cero, Paso a Paso

Vamos a implementar PCA completo con NumPy y aplicarlo al **dataset de dígitos escritos a mano** de scikit-learn.

El dataset tiene 1797 imágenes de 8×8 pixels cada una → cada imagen es un vector de 64 dimensiones. El reto de PCA es encontrar una representación en muchas menos dimensiones que capture la mayor varianza posible.

In [ ]:
# Cargamos el dataset
digits = load_digits()
X      = digits.data.astype(float)   # shape (1797, 64)
y      = digits.target               # etiquetas 0-9

print(f"Dataset: {X.shape[0]} imágenes, {X.shape[1]} features (pixeles)")
print(f"Clases: {np.unique(y)}")

# Visualizamos algunos dígitos
fig, axes = plt.subplots(2, 10, figsize=(13, 3))
for digito in range(10):
    idx = np.where(y == digito)[0][0]   # primera ocurrencia de cada dígito
    axes[0, digito].imshow(X[idx].reshape(8, 8), vmin=0, vmax=16)
    axes[0, digito].set_title(str(digito), fontsize=10)
    axes[0, digito].axis('off')
    idx2 = np.where(y == digito)[0][1]
    axes[1, digito].imshow(X[idx2].reshape(8, 8), vmin=0, vmax=16)
    axes[1, digito].axis('off')

plt.suptitle('Muestras del dataset de dígitos (8x8 pixels)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PCA desde cero — implementación completa
# ============================================================

class PCAdescratch:
    """
    PCA implementado con puras operaciones de NumPy.
    API intencionalmente parecida a sklearn.decomposition.PCA
    para que la comparación sea directa.
    """

    def __init__(self, n_components):
        self.n_components = n_components

    def fit(self, X):
        # Paso 1: centrar los datos
        self.mean_ = X.mean(axis=0)         # shape (n_features,)
        X_centrada = X - self.mean_          # broadcasting: (n,p) - (p,)

        # Paso 2: SVD de la matriz centrada
        # full_matrices=False → SVD "delgada", más eficiente
        U, S, Vt = np.linalg.svd(X_centrada, full_matrices=False)

        # Paso 3: guardar los componentes principales (filas de Vt)
        self.components_ = Vt[:self.n_components]   # shape (k, n_features)

        # Paso 4: varianza explicada por cada componente
        n = X.shape[0]
        varianzas = (S ** 2) / (n - 1)
        self.explained_variance_          = varianzas[:self.n_components]
        self.explained_variance_ratio_    = varianzas / varianzas.sum()
        self.explained_variance_ratio_    = self.explained_variance_ratio_[:self.n_components]

        return self

    def transform(self, X):
        """Proyectar X al espacio de los componentes principales."""
        X_centrada = X - self.mean_
        return X_centrada @ self.components_.T   # (n, p) @ (p, k) → (n, k)

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, X_reduced):
        """Reconstruir datos aproximados desde el espacio reducido."""
        return X_reduced @ self.components_ + self.mean_   # (n, k) @ (k, p) + (p,)


print("Clase PCAdescratch definida ✓")

In [ ]:
# Ajustamos PCA con todos los componentes para analizar la varianza
pca_full = PCAdescratch(n_components=64)
pca_full.fit(X)

print(f"Componentes calculados: {pca_full.components_.shape}")
print(f"Varianza explicada por PC1: {pca_full.explained_variance_ratio_[0]*100:.1f}%")
print(f"Varianza explicada por PC2: {pca_full.explained_variance_ratio_[1]*100:.1f}%")

---
# 7. Varianza Explicada y el Scree Plot

El **scree plot** muestra cuánta varianza explica cada componente principal. Es la herramienta estándar para decidir cuántos componentes conservar.

La **varianza acumulada** te dice: "con k componentes, explico X% de la varianza total del dataset".

In [ ]:
var_ratio     = pca_full.explained_variance_ratio_
var_acumulada = np.cumsum(var_ratio)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scree plot — varianza individual
axes[0].bar(range(1, 65), var_ratio * 100, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('% varianza explicada')
axes[0].set_title('Scree plot')
axes[0].grid(axis='y', alpha=0.3)

# Varianza acumulada
axes[1].plot(range(1, 65), var_acumulada * 100, color='darkorange', linewidth=2)
for umbral, color in [(0.80, 'green'), (0.90, 'red'), (0.95, 'purple')]:
    k_umbral = np.argmax(var_acumulada >= umbral) + 1
    axes[1].axhline(y=umbral*100, color=color, linestyle='--', alpha=0.6,
                    label=f'{umbral*100:.0f}% → k={k_umbral}')
    axes[1].axvline(x=k_umbral,   color=color, linestyle='--', alpha=0.6)

axes[1].set_xlabel('Número de componentes (k)')
axes[1].set_ylabel('% varianza acumulada')
axes[1].set_title('Varianza explicada acumulada')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Tabla resumen
print(f"{'k':>5}  {'varianza acumulada':>20}")
print("-" * 30)
for k in [2, 5, 10, 20, 30, 40, 50, 64]:
    print(f"{k:>5}  {var_acumulada[k-1]*100:>19.1f}%")

---
# 8. Visualización 2D del Espacio Latente

Si proyectamos el dataset de 64 dimensiones a solo **2 componentes principales**, podemos graficarlo. No captura toda la varianza (solo ~28%), pero debería verse estructura — los dígitos del mismo tipo deberían agruparse.

In [ ]:
pca_2d = PCAdescratch(n_components=2)
X_2d   = pca_2d.fit_transform(X)   # shape (1797, 2)

print(f"X original: {X.shape}  →  X_2d: {X_2d.shape}")
print(f"Varianza explicada: {pca_2d.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
# Scatter plot coloreado por dígito
colores = plt.cm.tab10(np.linspace(0, 1, 10))

fig, ax = plt.subplots(figsize=(9, 7))

for digito in range(10):
    mask = y == digito
    ax.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        color=colores[digito], label=str(digito),
        alpha=0.6, s=15
    )

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% varianza)')
ax.set_title('Dataset de dígitos proyectado a 2 componentes principales')
ax.legend(title='Dígito', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# Los componentes principales en sí mismos son vectores de 64 dimensiones
# → los podemos visualizar como imágenes 8x8 (son las "eigenfaces" de los dígitos)

fig, axes = plt.subplots(2, 8, figsize=(13, 4))

pca_viz = PCAdescratch(n_components=16)
pca_viz.fit(X)

for i in range(16):
    fila = i // 8
    col  = i  % 8
    axes[fila, col].imshow(pca_viz.components_[i].reshape(8, 8), cmap='RdBu_r')
    axes[fila, col].set_title(f'PC{i+1}', fontsize=8)
    axes[fila, col].axis('off')

plt.suptitle('Primeros 16 componentes principales — visualizados como imágenes 8x8',
             fontsize=11)
plt.tight_layout()
plt.show()

Cada componente principal es un "patrón de pixel" que, combinado con los demás, puede reconstruir cualquier dígito del dataset. Los primeros capturan las estructuras más comunes (trazo horizontal, vertical, curvas), y los últimos capturan detalles cada vez más finos.

---
# 9. Reconstrucción de Dígitos desde Componentes Principales

Así como con SVD reconstruimos imágenes con distintos k, aquí podemos reconstruir dígitos usando solo los primeros k componentes principales.

In [ ]:
# Tomamos un dígito de ejemplo de cada clase
ks_pca = [2, 5, 10, 20, 40, 64]
digito_ejemplo = 3
idx_ejemplo    = np.where(y == digito_ejemplo)[0][0]
muestra        = X[idx_ejemplo]   # shape (64,)

fig, axes = plt.subplots(1, len(ks_pca) + 1, figsize=(13, 2.5))

# Original
axes[0].imshow(muestra.reshape(8, 8), vmin=0, vmax=16)
axes[0].set_title('Original', fontsize=9)
axes[0].axis('off')

for i, k in enumerate(ks_pca):
    pca_k = PCAdescratch(n_components=k)
    pca_k.fit(X)

    muestra_reducida    = pca_k.transform(muestra.reshape(1, -1))
    muestra_reconstruida = pca_k.inverse_transform(muestra_reducida).ravel()

    var_exp = pca_k.explained_variance_ratio_.sum() * 100
    mse_rec = np.mean((muestra - muestra_reconstruida)**2)

    axes[i+1].imshow(muestra_reconstruida.reshape(8, 8), vmin=0, vmax=16)
    axes[i+1].set_title(f'k={k}\n{var_exp:.0f}% var', fontsize=9)
    axes[i+1].axis('off')

plt.suptitle(f'Reconstrucción del dígito "{digito_ejemplo}" con distintos k', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Hacemos lo mismo para un dígito de cada clase
k_mostrar = 10

pca_rec = PCAdescratch(n_components=k_mostrar)
pca_rec.fit(X)

fig, axes = plt.subplots(3, 10, figsize=(13, 4))

for digito in range(10):
    idx   = np.where(y == digito)[0][0]
    orig  = X[idx]
    red   = pca_rec.transform(orig.reshape(1, -1))
    rec   = pca_rec.inverse_transform(red).ravel()
    error = orig - rec

    axes[0, digito].imshow(orig.reshape(8, 8), vmin=0, vmax=16)
    axes[1, digito].imshow(rec.reshape(8, 8),  vmin=0, vmax=16)
    axes[2, digito].imshow(np.abs(error).reshape(8, 8), cmap='hot', vmin=0, vmax=8)

    if digito == 0:
        axes[0, digito].set_ylabel('Original',      fontsize=8)
        axes[1, digito].set_ylabel(f'Reconstruido\n(k={k_mostrar})', fontsize=8)
        axes[2, digito].set_ylabel('|Error|',       fontsize=8)

    axes[0, digito].set_title(str(digito), fontsize=10)
    for fila in range(3):
        axes[fila, digito].set_xticks([])
        axes[fila, digito].set_yticks([])

plt.suptitle(f'Original / Reconstruido (k={k_mostrar}) / Error absoluto', fontsize=11)
plt.tight_layout()
plt.show()

---
# 10. Comparación con scikit-learn

Nuestra implementación debería dar los mismos resultados que `sklearn.decomposition.PCA`. Verifiquémoslo.

In [ ]:
from sklearn.decomposition import PCA as SklearnPCA

k = 20

# Nuestra implementación
pca_nuestro = PCAdescratch(n_components=k)
X_nuestro   = pca_nuestro.fit_transform(X)

# Sklearn
pca_sklearn = SklearnPCA(n_components=k)
X_sklearn   = pca_sklearn.fit_transform(X)

# Los resultados deben ser iguales salvo signo
# (los eigenvectores son únicos salvo multiplicación por -1)
diff_var = np.abs(
    pca_nuestro.explained_variance_ratio_ -
    pca_sklearn.explained_variance_ratio_
).max()

print(f"Diferencia máxima en varianza explicada: {diff_var:.2e}")
print(f"¿Varianza explicada igual? {np.allclose(pca_nuestro.explained_variance_ratio_, pca_sklearn.explained_variance_ratio_)}")

# Las proyecciones deben ser iguales salvo signo en cada componente
# Comparamos la distancia entre puntos (que no depende del signo)
dist_nuestro = np.linalg.norm(X_nuestro[0] - X_nuestro[1])
dist_sklearn = np.linalg.norm(X_sklearn[0] - X_sklearn[1])
print(f"\nDistancia punto 0 a punto 1:")
print(f"  Nuestra implementación: {dist_nuestro:.6f}")
print(f"  Sklearn:                {dist_sklearn:.6f}")
print(f"  Iguales: {np.isclose(dist_nuestro, dist_sklearn)}")

In [ ]:
# Comparación visual: scatter 2D con ambas implementaciones
pca_2d_nuestro = PCAdescratch(n_components=2)
pca_2d_sklearn = SklearnPCA(n_components=2)

X_2d_nuestro = pca_2d_nuestro.fit_transform(X)
X_2d_sklearn = pca_2d_sklearn.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, X_proj, titulo in [
    (axes[0], X_2d_nuestro, 'PCA desde cero (NumPy)'),
    (axes[1], X_2d_sklearn, 'PCA sklearn')
]:
    for digito in range(10):
        mask = y == digito
        ax.scatter(X_proj[mask, 0], X_proj[mask, 1],
                   color=colores[digito], label=str(digito), alpha=0.5, s=12)
    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.legend(title='Dígito', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax.grid(alpha=0.2)

plt.suptitle('Comparación: nuestra implementación vs sklearn', fontsize=12)
plt.tight_layout()
plt.show()

print("\nLos plots son idénticos (salvo posible flip de ejes por diferencia de signo en eigenvectores).")

---
# Resumen

| Tema | Lo que aprendiste |
|------|-------------------|
| **Valores singulares** | Caen rápidamente → la mayoría de la información está en los primeros k |
| **Compresión SVD** | Guardar solo $U_k$, $S_k$, $V_k^T$ en lugar de la imagen completa |
| **SVD en RGB** | Aplicar SVD canal por canal de forma independiente |
| **Trade-off calidad** | PSNR > 30 dB = buena calidad con ~5-10x de compresión |
| **SVD ↔ PCA** | PCA es SVD de la matriz centrada — no son algoritmos distintos |
| **PCA desde cero** | Centrar → SVD → `components_` = filas de $V^T$ |
| **Varianza explicada** | $\sigma_i^2 / (n-1)$ mide cuánto aporta cada PC |
| **Scree plot** | Herramienta estándar para elegir k |
| **Espacio latente** | 64D → 2D proyectable y con estructura visible por clase |
| **inverse_transform** | Reconstruir datos desde el espacio comprimido |

El mismo principio que usamos aquí — encontrar las direcciones de mayor varianza y descartar el resto — aparece en redes neuronales (autoencoders), sistemas de recomendación (matrix factorization), procesamiento de lenguaje (LSA) y muchos otros algoritmos. Lo que cambia es cómo se parametriza la descomposición, no la idea central.